# HNSW Algorithm Implementation

## **Components**

#### *Search*: method for ANN search

#### *Insertion and Layer Assignment*: building the HNSW structure/inserting nodes for efficient searching



## **Search**

#### Suppose we have n nodes (A-E) in a layer, *query point Q*, and we want the 2 NNs (ef=2). We have the name of each node, their distances to Q, and their graph connections (who is connected to who at this layer). One of the n nodes, A, will be out *entry point*; we start here.

- *min-heap* : For *candidates*, keeps the smallest value at the top, index 0 = smallest dist --> explore next

- *max-heap* : For *found*, keeps the greatest value at the top (as a negative), index 0 = worst result (greatest distance), first to evict

#### We start a loop wherein we:
- (1) Pop the smallest valued node, A, from *candidates*, the node we explore from. 

- (2a) If this value is NOT worse than the *worst_found* (the largest value in *found*), then continue. 

- (2b) If this value is worse than the *worst_found* , discared the node.

#### Assuming (2a), and noting that (i) the first node has been popped out of *candidates* while (ii) staying in *found*, we check out A's neighbors, assuming B and C are designated as such. First we visit B: say the dist(Q,B)=2.0 < *worst_found* (5.0) and push B onto both heaps. 

#### Following (2a) again, we continue to A's other neighbor, C. Suppose dist(Q,C)=8.0 > *worst_found*(5.0). Since *found* is already full, C is discarded entirely.

#### *Now* we start the loop all over again! We pop B from *candidates*, compare with *worst_found* (still A), and explore B's neighbors, which could be [A, D, E]. Since A is already visited, we skip it and visit D.


#### Now that we've done this once in detail, I'll be less verbose from here on out.

#### dist(Q,D)=1.0 < *worst_found* (5.0) --> push D onto both heaps. *found* has 3 items but we pop the worst (A at 5.0) because ef=2.

#### Suppose dist(Q,E)=3.0 < *worst_found* (now 2.0, B). 3.0>2.0, so we discard E.

#### Loop again! Pop smallest from *candidates* (D at 1.0), 1.0 < *worst_found* (2.0), so we explore D's neighbors, say [B, E]. Both visited previously, skip both. Nothing pushed.

#### Candidates now empty, loop ends. Return *found*: [(1.0. D), (2.0, B)]. These are our 2 ANNs.

#### Generally: We consider a node (as a *candidate*), explore its neighbors, and continually update *found* until we have 2 nodes. Then we try continue to the next node, and so on.


In [9]:

import heapq

def search_layer(self, query, entry_points, ef, layer):
    visited = set(entry_points)
    candidates = []
    found = []

    for ep in entry_points:
        dist = self._dist(query, ep)
        heapq.heapppush(candidates, (dist, ep)) # push ep to candidates/min-heap w// distance
        heapq.heappush(found, (-dist, ep)) # push ep to found/max-heap w// distance
    
    while candidates:
        c_dist, c_id = heapq.heappop(candidates)
        worst_found = -found[0][0] # worst distance in found/max-heap
        
        if c_dist > worst_found:
            break
        
        for neighbor_id in self.graphs[layer].get(c_id, []):
            if neighbor_id in visited: 
                continue
            visited.add(neighbor_id)

            n_dist = self.graphs[layer].dist(query, self.vectors[neighbor_id])
            worst_found = -found[0][0]

            if n_dist < worst_found or len(found) < ef:
                heapq.heappush(candidates, (n_dist, neighbor_id))
                heapq.heappush(found, (-n_dist, neighbor_id))
                if len(found) > ef:
                    heapq.heappop(found)

    return sorted((-d, nid) for d, nid in found)




## **Select Neighbors**

#### The previous function is used for searching through layers to return a dynamic list of found NNs. This will be used not only for KNN Search but also for HNSW *construction*,  
#### When constructing our multilayer HNSW graph, we also need a means of connecting nodes. There are two methods we consider, the first naive and the second more robust:

#### (i) **Select-Neighbors (simple)**
- #### We want to connect a new node, Q, to neighbors. We use the output of *search_layer* (W, a list of *efConstruction* candidates, e.g. 50 candidates) to select the *M* nodes closest to Q and connect them to Q. Note the difference between *ef* and *efConstruction*; here, we use the latter. 
- #### **Problem:** Suppose we a candidate list sorted by distance to Q (A:1.0, B:1.2, C: 1.4, D:2.0, E:2.5). With parameter *M=3*, we would pick A,B, and C. However, A, B, and C might be clustered in the same region, and our node would end up with neighbors only pointing in one direction. This limits the maneuverability of the graph and ultimately the accuracy of our search.
- ### The heuristic below solves this problem!

#### (ii) **Select-Neighbors (heuristic)**
- #### Instead, a candidate from *M* only earns its place if it is closer to Q than it is to any already-selected neighbor. Assume the distances to Q listed above, but suppose the following are also true: dist(B-->A)=0.2 and dist(C-->A)=3.5. 
- #### Since B is NOT closer to Q than it is to A, it is disqualified. C is selected, however, and is pointing in a different direction.









In [ ]:
# Select Neighbors (Simple)
def select_neighbors_simple(self, candidates, M):
    return [nid for _, nid in candidates[:M]]

# Select Neighbors (Heuristic)
def select_neighbors_heuristic(self, candidates, M):
    selected = []
    for dist_to_query, cid in candidates:
        
        if len(selected) >= M:
            break
        is_useful = all(
            dist_to_query < self._dist(self.vectors[cid], self.vectors[s])
            for s in selected
        )

        if not selected or is_useful:
            selected.append(cid)
    return selected